# Meeting Action Agent (Claude Agent SDK)

A multi-agent system that turns a meeting transcript into a structured report.

**What this notebook builds:**
- One **orchestrator** (the boss) reads the transcript.
- It dispatches **three specialized subagents in parallel**:
  1. `decision-extractor` — finds concrete decisions
  2. `action-item-extractor` — finds who-does-what-by-when
  3. `follow-up-drafter` — drafts one email per attendee
- The orchestrator merges their outputs into one Markdown report.

**Why this design?** Each subagent has its own clean context window and one job to focus on. They never see each other's work, which keeps their prompts short and their answers sharp.


## Step 1 — Install the Claude Agent SDK

`claude-agent-sdk` is the official Python library that wraps the orchestrator + subagent logic. We also install `nest-asyncio` because Colab already has an event loop running, and the SDK needs to run its own async code on top of that.


In [ ]:
!pip install --quiet claude-agent-sdk nest-asyncio


## Step 2 — Set your Anthropic API key

You need an API key from https://console.anthropic.com/settings/keys (must start with `sk-ant-…`).

The cell below uses `getpass` so the key is hidden as you type — it won't appear in the notebook output if you push to GitHub.


In [ ]:
import os
from getpass import getpass

# Securely prompt for the API key — input is hidden as you type
os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key (starts with sk-ant-): ")
print("Key set. Length:", len(os.environ["ANTHROPIC_API_KEY"]))


## Step 3 — Define the four system prompts

A "system prompt" tells each AI agent what role to play and what rules to follow. We have **four** prompts:

| Prompt | Role |
|--------|------|
| `ORCHESTRATOR_PROMPT` | The boss — never analyzes the transcript itself; only dispatches subagents and stitches their outputs together |
| `DECISION_EXTRACTOR_PROMPT` | Finds concrete decisions the group made |
| `ACTION_ITEM_EXTRACTOR_PROMPT` | Finds tasks people committed to (owner + deadline) |
| `FOLLOW_UP_DRAFTER_PROMPT` | Writes a personalized email for each attendee |

Each prompt is just a Python string. The longer the prompt, the more specific the behavior — these are deliberately strict so the output format stays consistent.


In [ ]:
ORCHESTRATOR_PROMPT = """You are the orchestrator of a meeting analysis system.

Your job is to turn a raw meeting transcript into a structured, professional
meeting report. You do NOT analyze the transcript yourself. Instead, you
delegate to three specialized subagents IN PARALLEL via the Task tool:

  1. decision-extractor        - finds concrete decisions
  2. action-item-extractor     - finds tasks with owners + deadlines
  3. follow-up-drafter         - drafts personalized follow-up emails

WORKFLOW (follow exactly):
  Step 1. Invoke all three subagents IN PARALLEL by emitting three Task tool
          calls in a SINGLE message. Pass each subagent the full transcript.
  Step 2. After all three respond, merge their outputs into one Markdown
          report with these sections, in this order:
              # Meeting Report
              ## Decisions Made
              ## Action Items
              ## Follow-up Emails
  Step 3. Save the final report to output/meeting_report.md using the Write
          tool.
  Step 4. Reply to the user with a one-paragraph summary and the path to
          the saved report.

RULES:
  - Do NOT modify what the subagents returned. Stitch their Markdown together
    verbatim under the right section.
  - Do NOT invent facts. If a subagent reports "none found," preserve that.
  - Do NOT call subagents sequentially when they can run in parallel.
"""


DECISION_EXTRACTOR_PROMPT = """You are a decision extractor.

Given a meeting transcript, identify every concrete decision the group made.
A decision is something the team agreed to do or not do - NOT a discussion
item, NOT a suggestion, NOT a question.

OUTPUT FORMAT (Markdown, nothing else):

- **Decision:** <short title>
  - **Context:** <why this came up, one sentence>
  - **Resolution:** <what was decided, one sentence>
  - **Agreed by:** <who agreed, if clear from the transcript; otherwise "group">

If no decisions were made, output exactly:
> No decisions were made in this meeting.

Do not invent decisions. Do not add commentary outside the Markdown list.
"""


ACTION_ITEM_EXTRACTOR_PROMPT = """You are an action item extractor.

Given a meeting transcript, identify every action item - a task that a
specific person committed to doing after the meeting.

OUTPUT FORMAT (Markdown table, nothing else):

| Owner | Task | Deadline | Confidence |
|-------|------|----------|------------|
| <name> | <what they will do> | <date or "not specified"> | <high/medium/low> |

- "Confidence" reflects how clearly the task was assigned in the transcript.
  Use "high" if the owner explicitly accepted; "medium" if assigned but
  unconfirmed; "low" if only implied.
- If multiple people committed to the same task, list each on a separate row.

If no action items were committed to, output exactly:
> No action items identified.

Do not invent owners or deadlines. Do not add commentary outside the table.
"""


FOLLOW_UP_DRAFTER_PROMPT = """You are a follow-up email drafter.

Given a meeting transcript, identify each attendee who spoke and draft a
personalized follow-up email for each one.

Each email must:
  - Thank them for attending
  - Summarize what THEY specifically committed to (if anything)
  - Mention decisions relevant to their work
  - Be friendly, professional, and 3-5 sentences

OUTPUT FORMAT (Markdown, nothing else):

### To: <attendee name>
**Subject:** <relevant subject line>

<email body>

---

(Repeat the block for each attendee.)

If no attendees can be identified by name, output exactly:
> Could not identify named attendees in this transcript.

Do not invent attendees. Do not draft a generic group email.
"""

print("4 prompts loaded.")


## Step 4 — Configure the orchestrator and its three subagents

`build_options()` tells the SDK:
- **Which prompt** the orchestrator uses (role definition)
- **Which subagents exist**, what each one does, and which model it runs on
- **Which tools** the orchestrator is allowed to call (`Task` to dispatch subagents, `Write` to save the final report)
- The **working directory** (where files like `output/meeting_report.md` get saved)

Each subagent's `description` is what the orchestrator reads to decide *which* subagent to dispatch for a job — so each description must be one clear sentence.


In [ ]:
from claude_agent_sdk import (
    AgentDefinition,
    AssistantMessage,
    ClaudeAgentOptions,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    query,
)
from pathlib import Path

# Make sure the output folder exists before the run starts
PROJECT_ROOT = Path.cwd()
(PROJECT_ROOT / "output").mkdir(exist_ok=True)


def build_options() -> ClaudeAgentOptions:
    """Configure the orchestrator + 3 specialized subagents."""
    return ClaudeAgentOptions(
        # The orchestrator's role definition (the system prompt)
        system_prompt=ORCHESTRATOR_PROMPT,
        agents={
            # Subagent #1 — pulls out concrete decisions
            "decision-extractor": AgentDefinition(
                description=(
                    "Use when you need to extract concrete decisions made "
                    "during a meeting from a transcript."
                ),
                prompt=DECISION_EXTRACTOR_PROMPT,
                model="sonnet",  # use "haiku" for cheaper, "opus" for stronger
            ),
            # Subagent #2 — pulls out who-does-what-by-when
            "action-item-extractor": AgentDefinition(
                description=(
                    "Use when you need to extract action items with owners "
                    "and deadlines from a meeting transcript."
                ),
                prompt=ACTION_ITEM_EXTRACTOR_PROMPT,
                model="sonnet",
            ),
            # Subagent #3 — writes a personalized email per attendee
            "follow-up-drafter": AgentDefinition(
                description=(
                    "Use when you need to draft per-attendee follow-up emails "
                    "from a meeting transcript."
                ),
                prompt=FOLLOW_UP_DRAFTER_PROMPT,
                model="sonnet",
            ),
        },
        # Restrict the orchestrator: it may ONLY dispatch subagents and write files
        allowed_tools=["Task", "Write"],
        # Auto-approve file writes so the run doesn't pause for confirmation
        permission_mode="acceptEdits",
        # All relative paths resolve here
        cwd=str(PROJECT_ROOT),
    )


print("Options configured.")


## Step 5 — The sample meeting transcript

A realistic transcript with 5 attendees, several decisions, and 4-5 action items. Replace this with your own text to analyze a different meeting.


In [ ]:
TRANSCRIPT = """Weekly Product Sync - May 18, 2026 (10:00-10:35 AM)

Attendees: Priya (PM), Marcus (Engineering Lead), Sofia (Design), Ben (QA), Jordan (Data)

---

Priya: Morning everyone. Quick agenda - we need to lock down the v2.3 release scope,
go over the dashboard redesign feedback, and figure out what to do about the slow
sign-up flow. Let's start with v2.3. Marcus, where are we?

Marcus: We have three features ready - the bulk export, the saved filters, and the
new audit log. The mobile push notifications are still flaky. I'd recommend we ship
the first three and push notifications to v2.4.

Priya: I agree. Let's officially cut push notifications from v2.3 then. Ben, can you
have the regression pack done by Friday May 22?

Ben: Yes, I can have it done by Friday. I'll also include the new audit log scenarios
Marcus mentioned last week.

Priya: Perfect. Sofia, what did the users say about the dashboard redesign?

Sofia: Mostly positive. Eight out of ten testers preferred the new layout. The two
who didn't said the contrast on the secondary buttons was too low. I'd like to
update the design system tokens to fix that - probably 2-3 days of work.

Marcus: That's fine on our side. We don't have any blocking engineering work on the
dashboard until June.

Priya: Great. Sofia, please go ahead with the contrast fix. Let's say by May 27 so
we can include it in v2.3.

Sofia: I'll have it ready by the 27th.

Priya: Now the sign-up flow. Jordan, did you look at the funnel?

Jordan: I did. The drop-off is at step three - the email verification. Sixty-two
percent of new users abandon there. The average time to receive the verification
email is forty-one seconds, which is way above industry standard.

Marcus: That sounds like an SMTP queue issue. I can dig into it this week.

Jordan: I can put together a deeper funnel breakdown if that would help.

Priya: Yes please, that would help. Jordan, can you have the funnel analysis ready
by next Monday May 25? Marcus, sync with Jordan once you've identified the SMTP
root cause - I don't want to commit to a fix date until we know what we're dealing
with.

Marcus: Sounds good. I'll report back by Thursday May 21.

Jordan: I'll have the funnel breakdown by Monday.

Priya: One more thing - we should decide whether to keep weekly syncs or move to
biweekly. Thoughts?

Sofia: Honestly I think weekly is too much. We could do biweekly with a Slack
update in the off weeks.

Ben: I'd prefer to keep weekly. I find them useful.

Marcus: I lean toward biweekly too, but I don't feel strongly.

Priya: Let's stay weekly through the v2.3 release, then revisit. We can decide
again on June 10 after the release retro.

Priya: Good meeting everyone. Thanks for the focus.
"""

print(f"Transcript loaded ({len(TRANSCRIPT)} characters).")


## Step 6 — Helper functions for the live run

Two small helpers:

- **`build_user_prompt(transcript)`** — wraps the transcript in a clear user message. The transcript goes inside `<transcript>…</transcript>` tags so the model can easily tell instructions from data.
- **`_print_event(message)`** — prints a live trace as the run progresses. You'll see lines like `-> dispatching subagent: decision-extractor` so you can watch the parallel dispatch happen.


In [ ]:
def build_user_prompt(transcript: str) -> str:
    """Build the user message that goes to the orchestrator."""
    return f"""Analyze the meeting transcript below and produce the report
described in your instructions.

Remember: dispatch all three subagents IN PARALLEL (one message containing
three Task tool calls). Then merge their outputs and save to
output/meeting_report.md.

<transcript>
{transcript}
</transcript>
"""


def _print_event(message) -> None:
    """Pretty-print streamed events so you can follow the run live."""
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                # Stream model text without newline so chunks join up
                print(block.text, end="", flush=True)
            elif isinstance(block, ToolUseBlock):
                name = block.name
                if name == "Task":
                    # Task call = orchestrator delegating to a subagent
                    subagent = block.input.get("subagent_type", "?")
                    print(f"\n  -> dispatching subagent: {subagent}", flush=True)
                elif name == "Write":
                    # Write call = final report being saved to disk
                    path = block.input.get("file_path", "?")
                    print(f"\n  -> writing file: {path}", flush=True)
                else:
                    print(f"\n  -> tool: {name}", flush=True)
    elif isinstance(message, ResultMessage):
        # End-of-run summary
        print("\n\n--- run finished ---")
        if message.total_cost_usd is not None:
            print(f"cost: ${message.total_cost_usd:.4f} USD")
        if message.usage:
            print(f"tokens (input/output): "
                  f"{message.usage.get('input_tokens', 0)} / "
                  f"{message.usage.get('output_tokens', 0)}")


print("Helpers ready.")


## Step 7 — Run the agent system

Now the magic happens:

1. The orchestrator reads your transcript.
2. It emits **three `Task` tool calls in a single message** — that's parallel dispatch.
3. Each subagent runs in its own isolated context, processes the transcript, and returns its piece of the report.
4. The orchestrator stitches them together and writes `output/meeting_report.md`.

You'll see the live trace as it happens. Total cost is usually under $1.

> ⚠️ Colab already has an asyncio event loop running, so we apply `nest_asyncio` to let our async code run on top of it.


In [ ]:
import asyncio
import nest_asyncio

# Allow our async code to run inside Colab's existing event loop
nest_asyncio.apply()


async def analyze_meeting(transcript: str) -> None:
    """Run the full pipeline: orchestrator + 3 parallel subagents -> Markdown report."""
    options = build_options()
    user_prompt = build_user_prompt(transcript)

    print("Analyzing transcript...\n")
    # query() is an async generator — each yielded message is one streamed event
    async for message in query(prompt=user_prompt, options=options):
        _print_event(message)


# Run it! (top-level await works in Jupyter/Colab)
await analyze_meeting(TRANSCRIPT)


## Step 8 — View the generated report

The orchestrator saved the report to `output/meeting_report.md`. Let's display it as rendered Markdown.


In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

report_path = Path("output/meeting_report.md")
if report_path.exists():
    display(Markdown(report_path.read_text(encoding="utf-8")))
else:
    print("Report not found. Did the previous cell finish without errors?")


## Try your own transcript

Paste your own meeting transcript into the cell below and re-run the analyze cell:

```python
TRANSCRIPT = """...your transcript here..."""
await analyze_meeting(TRANSCRIPT)
```

### What to change

| To… | Edit… |
|-----|-------|
| Use a cheaper model | Change `model="sonnet"` to `model="haiku"` in `build_options()` |
| Use a stronger model | Change to `model="opus"` |
| Add a new subagent (e.g., sentiment analyzer) | Add another `AgentDefinition` in `build_options()` + a new prompt + mention it in `ORCHESTRATOR_PROMPT` |
| Change the report format | Edit `ORCHESTRATOR_PROMPT` (the section list under "WORKFLOW Step 2") |
